# 1. Memoria conversacional en un agente (LangGraph)

## Objetivos de Aprendizaje
- Separar **memoria de chat** (RA1) de **memoria de agente** (herramientas + hilo).
- Usar el recambio oficial: `checkpointer` + `thread_id` (ya lo viste en `RA1/IL1.1/4-langchain_memory.ipynb`).
- Ver por qué una lista `chat_history` actualizada a mano no escala.

El camino vigente es uno: `create_agent(..., checkpointer=InMemorySaver())`. El grafo guarda el estado del hilo. La API clásica (`AgentExecutor` + lista) quedó en `IL2.1/3-langchain-agent.ipynb` como contraste; aquí no se vuelve a ejecutar.


### 1. Instalación y configuración

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain-groq groq langgraph langchain requests python-dotenv
else:
    print("Entorno local: las dependencias ya las instaló uv sync.")


In [ ]:
import os
import re
import requests
from urllib.parse import quote

from langchain_groq import ChatGroq

try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Agente con tool calling nativo (create_agent / bind_tools).
# Por esta vía el modelo grande es el fiable; ver README de IL2.1.
MODELO = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

WIKIPEDIA_API_URL = "https://es.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY_URL = "https://es.wikipedia.org/api/rest_v1/page/summary"
WIKIPEDIA_HEADERS = {
    "User-Agent": "Curso-IA-DUOC/1.0 (notebook educativo; contacto: estudiante@example.com)"
}

try:
    llm = ChatGroq(model=MODELO, temperature=0, reasoning_effort="low")
    print("✅ LLM configurado.")
    print(f"Modelo: {MODELO}")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None


In [ ]:
from langchain_core.tools import tool


def _wikipedia_get_json(url, params=None):
    try:
        response = requests.get(url, params=params, headers=WIKIPEDIA_HEADERS, timeout=10)
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.JSONDecodeError:
        return None, "Wikipedia devolvió una respuesta vacía o no válida en JSON."
    except requests.exceptions.RequestException as e:
        return None, f"No se pudo consultar Wikipedia: {e}"


def _limitar_oraciones(texto, max_oraciones=2):
    oraciones = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(oraciones[:max_oraciones])


@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Ideal para personas, lugares o conceptos."""
    if not query or not query.strip():
        return "No se recibió un término de búsqueda para Wikipedia."

    search_data, error = _wikipedia_get_json(
        WIKIPEDIA_API_URL,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
            "utf8": 1,
        },
    )
    if error:
        return error

    results = search_data.get("query", {}).get("search", [])
    if not results:
        return f"No se encontró ninguna página para '{query}'."

    title = results[0]["title"]
    summary_data, error = _wikipedia_get_json(f"{WIKIPEDIA_SUMMARY_URL}/{quote(title)}")
    if error:
        return error

    extract = summary_data.get("extract")
    if not extract:
        return f"Wikipedia no entregó un resumen disponible para '{title}'."

    return _limitar_oraciones(extract, max_oraciones=2)


tools = [get_wikipedia_summary]
print("✅ Herramienta Wikipedia lista. El docstring ES lo que ve el modelo.")


### 2. El checkpointer ES la memoria

No pasas `chat_history`. Pasas el mismo `thread_id`. El segundo turno ve al primero, incluidas las llamadas a Wikipedia.


In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agente = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Eres un asistente que busca en Wikipedia cuando necesita un dato.",
    checkpointer=InMemorySaver(),
)
hilo = {"configurable": {"thread_id": "saturno"}}


def preguntar(texto: str) -> str:
    salida = agente.invoke({"messages": [HumanMessage(content=texto)]}, hilo)
    return salida["messages"][-1].content


print("Turno 1 — sin contexto previo")
print(preguntar("Háblame del planeta Saturno"))
print("\nTurno 2 — no nombra Saturno")
print(preguntar("¿Y de qué están hechos sus anillos?"))


Si el turno 2 habla de hielo y roca, el checkpointer hizo su trabajo. Cambia el `thread_id` y el agente **olvida**: así se aíslan sesiones (un alumno, un ticket, un paciente).


In [ ]:
otro = {"configurable": {"thread_id": "otro-usuario"}}
amnesia = agente.invoke(
    {"messages": [HumanMessage(content="¿De qué están hechos sus anillos?")]},
    otro,
)
print("Otro hilo (no debería saber de Saturno):")
print(amnesia["messages"][-1].content)


## Conclusiones

- **Stateless** = cada llamada nace amnésica. La memoria es lo que convierte una API en un asistente.
- **Vigente:** `checkpointer` + `thread_id`. El grafo persiste el estado. Es el mismo mecanismo de RA1, ahora con herramientas.
- **Límite:** un buffer eterno se come la cuota. Eso se resuelve en el siguiente notebook (ventana y resumen).

Siguiente: `2-memory-agent-advanced.ipynb`. Después, herramientas externas y MCP en `3-herramientas-externas.ipynb` (SDK crudo: ese fundamento no se toca).
